# 006 — LoRA Fine-Tuning of Qwen2.5-1.5B for AI Text Detection

This notebook fine-tunes `Qwen/Qwen2.5-1.5B` (a 1.5B-parameter causal language model) using **LoRA (Low-Rank Adaptation)** on the PAN 2026 AI-generated text detection task.

**What it does:**
1. Downloads PAN2025 training & validation data from Google Drive
2. Loads Qwen2.5-1.5B as a sequence classifier with a classification head
3. Applies LoRA adapters (only ~0.2% of parameters are trainable)
4. Trains on GPU with bf16 for 3 epochs
5. Evaluates on the validation set
6. Pushes the LoRA adapter to HuggingFace Hub

**Why a causal LM?** Decoder-only models are pre-trained autoregressively, giving them an inherent model of how text is generated. This inductive bias may help distinguish human-written from machine-generated text.

**After running this notebook**, you can load the adapter on your laptop for local evaluation.

In [1]:
# =====================================================
# CELL 0 — Environment config (MUST be first)
# =====================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["DATASETS_DISABLE_PROGRESS"] = "1"
print("Environment configured. Progress bars disabled.")

Environment configured. Progress bars disabled.


In [2]:
# =====================================================
# CELL 1 — Install dependencies
# =====================================================
!pip install -q peft gdown bitsandbytes
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 38.8 MB/s eta 0:00:00:00:0100:01
Dependencies installed.


In [3]:
# =====================================================
# CELL 2 — Imports & device check
# =====================================================
import json
import numpy as np
import pandas as pd
import torch
import gdown
from datasets import Dataset, disable_progress_bar
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    AutoConfig,
    TrainingArguments,
    Trainer,
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import roc_auc_score, f1_score

disable_progress_bar()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Constants ──────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAX_LENGTH = 512

# ── LoRA hyper-parameters ──────────────────────────
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

Using device: cuda
GPU: NVIDIA L4


In [4]:
# =====================================================
# CELL 3 — Download data from Google Drive
# =====================================================
train_id = '1k84YY0p8JTuTE40yNEkgzsdaSxt_1nl-'
val_id   = '12szC1TcNPN9KULPNjTZsEyG8RafxfsJy'

BASE = "/content/data/"
os.makedirs(BASE, exist_ok=True)

TRAIN_FILE = BASE + "train.jsonl"
VAL_FILE   = BASE + "val.jsonl"

print("Downloading train...")
gdown.download(f'https://drive.google.com/uc?id={train_id}', TRAIN_FILE, quiet=True)
print("Downloading val...")
gdown.download(f'https://drive.google.com/uc?id={val_id}', VAL_FILE, quiet=True)
print("Downloads complete.")

Downloads complete.


In [5]:
# =====================================================
# CELL 4 — Load & inspect data
# =====================================================
def load_jsonl(path):
    with open(path, 'r') as f:
        return pd.DataFrame([json.loads(l) for l in f])

train_df = load_jsonl(TRAIN_FILE)
val_df   = load_jsonl(VAL_FILE)
print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}")
print(f"Label distribution (train): {dict(train_df['label'].value_counts())}")

Train: 23,707  |  Val: 3,589
Label distribution (train): {1: np.int64(14606), 0: np.int64(9101)}


In [ ]:
# =====================================================
# CELL 5 — Tokenize datasets
# =====================================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Causal LMs need a pad token; use eos_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Left-padding so the classification head reads the last
# real token, not a padding token
tokenizer.padding_side = "left"
print(f"Tokenizer loaded. Pad token: {tokenizer.pad_token!r}, padding side: {tokenizer.padding_side}")

def tokenize_df(df):
    ds = Dataset.from_pandas(df[['text', 'label']])
    def tok(ex):
        return tokenizer(
            ex['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH,
        )
    return (
        ds.map(tok, batched=True, batch_size=1000, num_proc=1, remove_columns=['text'])
          .rename_column('label', 'labels')
    )

print("Tokenizing train...")
train_dataset = tokenize_df(train_df)
print("Tokenizing val...")
val_dataset   = tokenize_df(val_df)
print("Done.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Tokenizer loaded. Pad token: '<|endoftext|>', padding side: left
Tokenizing train...


In [14]:
# =====================================================
# CELL 6 — Load base model + wrap with LoRA
# =====================================================
print("Loading base model...")

# Configure pad token and classification in the model config
# (Qwen2.5 uses GenericForSequenceClassification which requires
#  num_labels and problem_type to be set on the config, not as kwargs)
config = AutoConfig.from_pretrained(MODEL_NAME)
config.pad_token_id = config.eos_token_id
config.num_labels = 2
config.problem_type = "single_label_classification"

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    torch_dtype=torch.bfloat16,
)

# Gradient checkpointing for memory efficiency
base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

print(f"Base model parameters: {base_model.num_parameters():,}")

# ── LoRA config ────────────────────────────────────
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading base model...


Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base model parameters: 1,543,717,376
trainable params: 2,182,144 || all params: 1,545,899,520 || trainable%: 0.1412


In [15]:
# =====================================================
# CELL 7 — Training setup
# =====================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    preds = np.argmax(logits, axis=-1)
    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except Exception:
        auc = 0.5
    return {
        'roc_auc': auc,
        'f1': f1_score(labels, preds),
        'accuracy': (preds == labels).mean(),
    }

use_cuda = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="/content/lora_checkpoints",
    # ── Device ─────────────────────────────────────
    use_cpu=not use_cuda,
    bf16=use_cuda,
    # ── Batch size ─────────────────────────────────
    per_device_train_batch_size=8 if use_cuda else 2,
    per_device_eval_batch_size=16 if use_cuda else 4,
    gradient_accumulation_steps=2 if use_cuda else 8,
    # ── Optimizer ──────────────────────────────────
    learning_rate=2e-4,          # Higher LR for LoRA
    weight_decay=0.01,
    warmup_steps=100,
    # ── Schedule ───────────────────────────────────
    num_train_epochs=3,
    # ── Evaluation ─────────────────────────────────
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=50,
    # ── Checkpointing ──────────────────────────────
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    # ── Misc ───────────────────────────────────────
    report_to="none",
    disable_tqdm=True,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
print("Trainer ready.")

Trainer ready.


In [16]:
# =====================================================
# CELL 8 — Train!
# =====================================================
print("Starting LoRA fine-tuning of Qwen2.5-1.5B...")
trainer.train()
print("Training complete.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting LoRA fine-tuning of Qwen2.5-1.5B...
{'loss': '8.451', 'grad_norm': '48.59', 'learning_rate': '9.8e-05', 'epoch': '0.03374'}
{'loss': '0.3979', 'grad_norm': '0.05601', 'learning_rate': '0.000198', 'epoch': '0.06748'}
{'loss': '0.2744', 'grad_norm': '0.1084', 'learning_rate': '0.0001977', 'epoch': '0.1012'}
{'loss': '0.07041', 'grad_norm': '0.2615', 'learning_rate': '0.0001954', 'epoch': '0.135'}
{'eval_loss': '0.1216', 'eval_roc_auc': '0.9958', 'eval_f1': '0.9851', 'eval_accuracy': '0.9811', 'eval_runtime': '154.8', 'eval_samples_per_second': '23.18', 'eval_steps_per_second': '1.453', 'epoch': '0.135'}
{'loss': '0.182', 'grad_norm': '0.1091', 'learning_rate': '0.0001931', 'epoch': '0.1687'}
{'loss': '0.105', 'grad_norm': '0.00762', 'learning_rate': '0.0001908', 'epoch': '0.2024'}
{'loss': '0.2017', 'grad_norm': '0.3462', 'learning_rate': '0.0001885', 'epoch': '0.2362'}
{'loss': '0.08791', 'grad_norm': '0.002646', 'learning_rate': '0.0001862', 'epoch': '0.2699'}
{'eval_loss': '0

In [17]:
# =====================================================
# CELL 9 — Final evaluation on validation set
# =====================================================
results = trainer.evaluate()
print("Validation results:")
for k, v in results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

{'eval_loss': '0.01375', 'eval_roc_auc': '0.9999', 'eval_f1': '0.9981', 'eval_accuracy': '0.9975', 'eval_runtime': '154.5', 'eval_samples_per_second': '23.24', 'eval_steps_per_second': '1.457', 'epoch': '3'}
Validation results:
  eval_loss: 0.0138
  eval_roc_auc: 0.9999
  eval_f1: 0.9981
  eval_accuracy: 0.9975
  eval_runtime: 154.4508
  eval_samples_per_second: 23.2370
  eval_steps_per_second: 1.4570
  epoch: 3.0000


In [20]:
# =====================================================
# CELL 10 — Save adapter locally + push to HuggingFace
# =====================================================
from huggingface_hub import HfApi, login

# ── CONFIGURE THESE ────────────────────────────────
HF_TOKEN    = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"  # <- Replace with your token
HF_USERNAME = "hersheys-baklava"                         # <- Replace with your HF username
REPO_NAME   = "qwen-lora-pan2026"                        # <- Repo name for the adapter
# ──────────────────────────────────────────────────

login(token=HF_TOKEN)

ADAPTER_DIR = "/content/lora_adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Save LoRA adapter weights (small! typically < 10 MB)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Save training metrics alongside the adapter
with open(f"{ADAPTER_DIR}/training_metrics.json", 'w') as f:
    json.dump(results, f, indent=2)

# Also save the LoRA config as a readable summary
lora_summary = {
    "base_model": MODEL_NAME,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": LORA_TARGET_MODULES,
    "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "total_params": sum(p.numel() for p in model.parameters()),
    "max_length": MAX_LENGTH,
    "num_epochs": 3,
    "learning_rate": 2e-4,
}
with open(f"{ADAPTER_DIR}/lora_config_summary.json", 'w') as f:
    json.dump(lora_summary, f, indent=2)

print(f"Adapter saved locally to {ADAPTER_DIR}")
print(f"Adapter size: {sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)) / 1024:.1f} KB")

Adapter saved locally to /content/lora_adapter
Adapter size: 19696.6 KB


In [21]:
# =====================================================
# CELL 11 — Upload adapter to HuggingFace Hub
# =====================================================
api = HfApi()
repo_id = f"{HF_USERNAME}/{REPO_NAME}"

# Create repo (set private=False to make it public)
api.create_repo(repo_id=repo_id, exist_ok=True, private=True)

# Upload the adapter folder
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=repo_id,
    commit_message="Upload LoRA adapter for Qwen2.5-1.5B AI text detection",
)

print(f"\n=== Adapter uploaded to: https://huggingface.co/{repo_id} ===")
print(f"\nTo load on your laptop for evaluation:\n")
print(f"  from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig")
print(f"  from peft import PeftModel")
print(f"")
print(f"  config = AutoConfig.from_pretrained('{MODEL_NAME}')")
print(f"  config.pad_token_id = config.eos_token_id")
print(f"  config.num_labels = 2")
print(f"  config.problem_type = 'single_label_classification'")
print(f"  base = AutoModelForSequenceClassification.from_pretrained('{MODEL_NAME}', config=config)")
print(f"  model = PeftModel.from_pretrained(base, '{repo_id}')")
print(f"  model = model.merge_and_unload()  # merge for faster CPU inference")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{repo_id}')")
print(f"  tokenizer.padding_side = 'left'  # important for causal LM classification")


=== Adapter uploaded to: https://huggingface.co/hersheys-baklava/qwen-lora-pan2026 ===

To load on your laptop for evaluation:

  from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
  from peft import PeftModel

  config = AutoConfig.from_pretrained('Qwen/Qwen2.5-1.5B')
  config.pad_token_id = config.eos_token_id
  config.num_labels = 2
  config.problem_type = 'single_label_classification'
  base = AutoModelForSequenceClassification.from_pretrained('Qwen/Qwen2.5-1.5B', config=config)
  model = PeftModel.from_pretrained(base, 'hersheys-baklava/qwen-lora-pan2026')
  model = model.merge_and_unload()  # merge for faster CPU inference
  tokenizer = AutoTokenizer.from_pretrained('hersheys-baklava/qwen-lora-pan2026')
  tokenizer.padding_side = 'left'  # important for causal LM classification


---

## Next Steps (on your laptop)

Once the adapter is on HuggingFace, run the local evaluation script:

```bash
cd pan-2026/pan/src/006-lora-baseline
python3 evaluate.py \
    --adapter-path <HF_USERNAME>/<REPO_NAME>
```

Or load it manually in Python:

```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from peft import PeftModel

config = AutoConfig.from_pretrained("Qwen/Qwen2.5-1.5B")
config.pad_token_id = config.eos_token_id
config.num_labels = 2
config.problem_type = "single_label_classification"

base = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-1.5B", config=config
)
model = PeftModel.from_pretrained(base, "<HF_USERNAME>/<REPO_NAME>")
model = model.merge_and_unload()

tokenizer = AutoTokenizer.from_pretrained("<HF_USERNAME>/<REPO_NAME>")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # critical for causal LM classification

# Now use model + tokenizer for inference
```

The evaluation script (`evaluate.py`) will:
1. Load the adapter from HuggingFace (or a local path)
2. Merge LoRA weights into the base Qwen2.5-1.5B model
3. Run chunked inference on PAN2025 val + HC3
4. Compute all 5 PAN metrics (ROC-AUC, Brier, F1, C@1, F0.5u)
5. Print a comparison table against all prior experiments